## 1. The question and the data
This project analyses late deliveries using the Olist dataset: 99,441 real orders from a Brazilian online marketplace. The question: where and why are deliveries late, and how does lateness affect customer reviews?

In [ ]:
import pandas as pd
orders = pd.read_csv('/content/olist_orders_dataset.csv')
customers = pd.read_csv('/content/olist_customers_dataset.csv')
items = pd.read_csv('/content/olist_order_items_dataset.csv')
sellers = pd.read_csv('/content/olist_sellers_dataset.csv')
reviews = pd.read_csv('/content/olist_order_reviews_dataset.csv')
len(orders)

99441

## 2. Data checks
I checked the orders for gaps and errors. Most missing values were from orders that hadn’t been delivered yet, which is normal. 8 orders were marked “delivered” but had no delivery date, so I excluded them, leaving 96,470 orders to measure.

In [ ]:
display(orders['order_status'].value_counts())
display(orders.isna().sum())
# delivered orders with no delivery date (a data problem)
display(orders[(orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isna())])
delivered = orders[(orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].notna())].copy()
delivered['order_delivered_customer_date'] = pd.to_datetime(delivered['order_delivered_customer_date'])
delivered['order_estimated_delivery_date'] = pd.to_datetime(delivered['order_estimated_delivery_date'])
len(delivered)

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


96470

## 3. Defining "late"
An order counts as late if the customer received it after the estimated delivery date. The estimated date has no time, so comparing full timestamps made same-day deliveries look late (8.11%). Comparing dates only gives the true picture: 6.77% of orders were late.

In [ ]:
with_time = (delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']).mean() * 100
delivered['late'] = delivered['order_delivered_customer_date'].dt.normalize() > delivered['order_estimated_delivery_date']
print('Late % using timestamps:', round(with_time, 2))
print('Late % using dates only:', round(delivered['late'].mean() * 100, 2))

Late % using timestamps: 8.11
Late % using dates only: 6.77


## 4. What: overall late %
Overall, 6.77% of delivered orders arrived late.

## 5. Where: late % by state
By late %, the worst state is AL (21.4%), but it has only 397 orders, so fixing it would change little overall. Looking at volume instead, RJ is the second-biggest market (12,350 orders) and is late 12.1% of the time, nearly three times São Paulo’s 4.5%. Improving Rio would have by far the biggest impact.

In [ ]:
merged = delivered.merge(customers, on='customer_id', how='left')
print('Rows:', len(merged), '| Missing states:', merged['customer_state'].isna().sum())
by_state = merged.groupby('customer_state').agg(
    orders=('order_id', 'count'),
    late_pct=('late', 'mean'))
by_state['late_pct'] = by_state['late_pct'] * 100
display(by_state.sort_values('late_pct', ascending=False).head(5))
by_state.sort_values('orders', ascending=False).head(5)

Rows: 96470 | Missing states: 0


,orders,late_pct
customer_state,,
AL,397,21.410579
MA,717,17.433752
SE,335,15.223881
PI,476,13.865546
CE,1279,13.760751


,orders,late_pct
customer_state,,
SP,40494,4.494493
RJ,12350,12.105263
MG,11354,4.571076
RS,5344,6.081587
PR,4923,4.042251


## 6. Why: same-state vs cross-state sellers
Orders with several items appeared more than once, so I kept one seller per order (for the 1,278 orders with more than one seller, the first was used). I then compared orders where the seller was in the same state as the customer with those where they weren’t. Cross-state orders were late almost twice as often: 8.0% vs 4.5%.

In [ ]:
item_sellers = items.merge(sellers, on='seller_id', how='left')
sellers_per_order = item_sellers.groupby('order_id')['seller_id'].nunique()
print('Orders with more than one seller:', (sellers_per_order > 1).sum())
one_seller = item_sellers.drop_duplicates('order_id')[['order_id', 'seller_state']]
merged1 = merged.merge(one_seller, on='order_id', how='left')
print('Rows:', len(merged1), '| Missing seller states:', merged1['seller_state'].isna().sum())
merged1['same_state'] = merged1['customer_state'] == merged1['seller_state']
by_same = merged1.groupby('same_state').agg(
    orders=('order_id', 'count'),
    late_pct=('late', 'mean'))
by_same['late_pct'] = by_same['late_pct'] * 100
by_same

Orders with more than one seller: 1278
Rows: 96470 | Missing seller states: 0


,orders,late_pct
same_state,,
False,61769,8.047726
True,34701,4.504193


## 7. The cost: review scores
I then looked at whether late deliveries affect customer reviews. 551 orders had more than one review, so I averaged their scores. Late orders averaged 2.27 stars compared with 4.29 for on-time orders, and 62% of late orders received a bad review (1–2 stars), against 9% of on-time ones.

In [ ]:
print('Orders with more than one review:', reviews['order_id'].duplicated().sum())
display(reviews['review_score'].value_counts())
order_scores = reviews.groupby('order_id', as_index=False).agg(
    review_score=('review_score', 'mean'))
merged2 = merged1.merge(order_scores, on='order_id', how='left')
print('Rows:', len(merged2), '| Orders with no review:', merged2['review_score'].isna().sum())
by_late = merged2.groupby('late').agg(
    orders=('order_id', 'count'),
    avg_score=('review_score', 'mean'))
display(by_late)
reviewed = merged2[merged2['review_score'].notna()].copy()
reviewed['bad_review'] = reviewed['review_score'] <= 2
reviewed.groupby('late')['bad_review'].mean() * 100

Orders with more than one review: 551


,count
review_score,
5,57328
4,19142
1,11424
3,8179
2,3151


Rows: 96470 | Orders with no review: 646


,orders,avg_score
late,,
False,89936,4.290589
True,6534,2.271823


,bad_review
late,
False,9.231578
True,62.356997


## Export for Power BI
The cleaned, joined table is saved as a CSV and used as the source for the Power BI dashboard.

In [ ]:
merged2.to_csv('olist_delivered_clean.csv', index=False)

## 8. Summary and assumptions

**Findings**
- **What:** 6.77% of delivered orders arrived late.
- **Where:** AL has the worst late rate (21.4%) but only 397 orders. RJ, the second-biggest market, is late 12.1% of the time, nearly 3× São Paulo's 4.5%.
- **Why:** cross-state orders are late almost twice as often as same-state ones (8.0% vs 4.5%).
- **Cost:** late orders average 2.27 stars vs 4.29 for on-time orders, and 62% of them get a bad review (vs 9%).

**Recommendation**
Focus on large markets with high late rates, such as Rio, rather than small states with extreme percentages. A clear definition of "late" was essential to measuring how lateness affects customer satisfaction.

**Assumptions**
- 8 delivered orders with no delivery date were excluded.
- For 1,278 orders with more than one seller, the first seller was used.
- 551 orders with several reviews had their scores averaged.
- 646 orders with no review were left out of the review figures.